# GenAI Platform — Demo

Este notebook demonstra a plataforma camada por camada.

```
ADKRuntime  ←  mesmo Tool, zero mudança de business logic
    ↓
CandidateSearchTool   execute() + schema()
    ↓
CandidateRetriever
    ↓
EmbeddingProvider     GeminiEmbeddingProvider
VectorStore           FAISSVectorStore
```

**Regra de validação:** trocar qualquer provider muda só o wiring. Nunca a business logic.

## 0. Setup

In [ ]:
!pip install -q -U google-genai google-adk pydantic python-dotenv faiss-cpu requests

In [ ]:
import os
import sys

# Clone repo se ainda não existir
if not os.path.exists('/content/agent_workshop'):
    !git clone https://github.com/Willblemos2/agent_workshop.git /content/agent_workshop

repo_root = '/content/agent_workshop'
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

In [ ]:
# Carrega a API key do Colab Secrets (menu lateral → chave 🔑 → GOOGLE_API_KEY)
from google.colab import userdata
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')

print('API key configurada.')

## 1. Dataset

Carregamos o dataset já processado — embeddings Gemini (3072 dim) pré-computados.
Não há chamada de API nesta etapa.

In [ ]:
from src.shared.dataset import load_processed_candidates

candidates_df = load_processed_candidates()

print(f'Candidatos carregados: {len(candidates_df)}')
candidates_df[['candidate_id', 'first_name', 'last_name', 'content']].head(3)

## 2. Foundation — Vector Store

Criamos o índice FAISS e inserimos os candidatos usando os embeddings pré-computados.

In [ ]:
from src.foundation.vector_store import FAISSVectorStore, build_records
from src.foundation.vector_store.schemas import CANDIDATE_SCHEMA

store = FAISSVectorStore()
store.create_index(index_name='candidates_index', schema=CANDIDATE_SCHEMA)

records = build_records(
    rows=candidates_df.to_dict(orient='records'),
    id_field='candidate_id',
)
store.insert(index_name='candidates_index', records=records)

print(f'{len(records)} candidatos indexados no FAISS.')

## 3. Foundation — Embedding Provider

O provider é injetado — o retriever não sabe qual está usando.

In [ ]:
from src.foundation.embeddings import GeminiEmbeddingProvider

embedding_provider = GeminiEmbeddingProvider()

sample = embedding_provider.embed('Senior Python engineer')
print(f'Embedding gerado: {len(sample)} dimensões')

## 4. Application — Retriever

`CandidateRetriever` não conhece Gemini, não conhece FAISS.
Recebe os dois por injeção — troca de provider não muda esta camada.

In [ ]:
from src.applications.retrieval import CandidateRetriever

retriever = CandidateRetriever(
    vector_store=store,
    embedding_provider=embedding_provider,
)

results = retriever.search('Python developer with machine learning experience', top_k=3)

for r in results:
    print(f'[{r.score:.4f}] {r.first_name} {r.last_name} — {r.content_preview[:80]}...')

## 5. Tool Layer

O `Tool` expõe dois contratos:
- `execute()` — roda a capability
- `schema()` — descreve a tool para qualquer runtime

O runtime lê `schema()` para registrar a tool no framework. Nunca toca a business logic.

In [ ]:
from src.tools import CandidateSearchTool

search_tool = CandidateSearchTool(retriever=retriever)

# schema() — o que qualquer runtime vai ler
schema = search_tool.schema()
print(f'Tool: {schema.name}')
print(f'Descrição: {schema.description}')
print(f'Parâmetros:')
for p in schema.parameters:
    req = 'obrigatório' if p.required else 'opcional'
    print(f'  - {p.name} ({p.type}, {req}): {p.description}')

In [ ]:
# execute() — chamada direta, sem agent
result = search_tool.execute(query='data scientist with NLP skills', top_k=3)

if result.error:
    print(f'Erro: {result.error}')
else:
    for c in result.content:
        print(f"[{c['score']:.4f}] {c['first_name']} {c['last_name']}")

## 6. Runtime — ADK Agent

O mesmo `CandidateSearchTool` agora roda dentro do ADK.
O `ADKRuntime` lê `tool.schema()` e converte para o formato do ADK automaticamente.

Zero mudança na tool, zero mudança no retriever.

In [ ]:
from src.runtime import ADKRuntime

runtime = ADKRuntime(
    model='gemini-2.5-flash',
    instruction='You are a recruitment assistant. Use the candidate_search tool to find relevant candidates and summarize your findings clearly.',
)

response = await runtime.run(
    input='Find me 3 software engineers with cloud infrastructure experience.',
    tools=[search_tool],
)

print(response)

In [ ]:
# Stream — mesma tool, mesma lógica, output em tempo real
async for chunk in runtime.stream(
    input='Who are the best candidates for a senior data engineering role?',
    tools=[search_tool],
):
    print(chunk, end='', flush=True)

## 7. Harness — Troca de Provider

Validação arquitetural: trocar FAISS por Milvus não muda nenhuma linha de business logic.

In [ ]:
!pip install -q "pymilvus>=2.4.0,<2.5"

In [ ]:
from src.foundation.vector_store import MilvusVectorStore

# Única linha que muda — wiring
milvus_store = MilvusVectorStore()
milvus_store.create_index(index_name='candidates_index', schema=CANDIDATE_SCHEMA)
milvus_store.insert(index_name='candidates_index', records=records)

# Retriever, Tool e Runtime — idênticos
milvus_retriever = CandidateRetriever(
    vector_store=milvus_store,
    embedding_provider=embedding_provider,
)
milvus_tool = CandidateSearchTool(retriever=milvus_retriever)

milvus_results = milvus_retriever.search('Python developer with machine learning experience', top_k=3)

print('Resultados via Milvus:')
for r in milvus_results:
    print(f'[{r.score:.4f}] {r.first_name} {r.last_name}')